# 02 - Experiments

Fixed-parameter experiments that justify the design of the final pipeline (`python -m src.train`).

Fraud is rare and PaySim is simulated, so treat small differences as ties and large differences as design signals.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.config import DATA_PATH, FPR_BUDGETS, METRICS_PATH, SEED, TARGET
from src.data import load_data, time_split
from src.evaluate import operating_point, score_summary, threshold_for_fpr
from src.features import build_features, feature_columns
from src.models import downsample_negatives, make_xgb, prior_correct

NROWS = None   # for example 1_000_000 for a faster, smaller run
NEG_FRAC = 0.2
PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8)

df = load_data(DATA_PATH, nrows=NROWS)
X = build_features(df)
y = df.loc[X.index, TARGET].values
step = df.loc[X.index, "step"].values
amount = df.loc[X.index, "amount"].values
tr, va, te = time_split(step)
print({n: (int(m.sum()), int(y[m].sum())) for n, m in [("train", tr), ("val", va), ("test", te)]}, "(rows, fraud)")

In [ ]:
def fit_eval(train_mask, val_mask, test_mask, cols, label):
    X_tr, y_tr = downsample_negatives(X.loc[train_mask, cols], y[train_mask], NEG_FRAC)
    model = make_xgb(**PARAMS).fit(X_tr, y_tr)
    p_val = prior_correct(model.predict_proba(X.loc[val_mask, cols])[:, 1], NEG_FRAC)
    p_te = prior_correct(model.predict_proba(X.loc[test_mask, cols])[:, 1], NEG_FRAC)
    s = score_summary(y[test_mask], p_te, y[val_mask], p_val, amount[test_mask])
    return {"setup": label, "PR-AUC": s["pr_auc"], "ROC-AUC": s["roc_auc"],
            **{f"recall @ {b:.2%} FPR": s[f"recall_at_fpr_{b}"] for b in FPR_BUDGETS}}

## Experiment 1: random split vs time-based split

Same model, same features, only the split differs. A random split lets the model train on transactions from the same period it is tested on, which flatters it. In production the model only ever sees the past.

In [ ]:
rng = np.random.default_rng(SEED)
u = rng.random(len(y))
r_tr, r_va, r_te = u < 0.64, (u >= 0.64) & (u < 0.80), u >= 0.80
cols = feature_columns(X.columns, "pretxn")
pd.DataFrame([
    fit_eval(r_tr, r_va, r_te, cols, "random split"),
    fit_eval(tr, va, te, cols, "time-based split"),
]).round(4)

If the time-based row is clearly worse, the random split overstated performance. If they are close, the simulation has little drift. Either way, the time-based split is the honest one and is what the pipeline uses.

## Experiment 2: how much do the balance columns matter?

- `full`: all features, including balances after the transaction.
- `pretxn`: only what is known before the transaction completes (the final model).
- `no_balance`: no balance information at all.

Some descriptions of PaySim warn that detected fraud is cancelled, so the balance-after columns can encode the label. A large gap between `full` and `pretxn` is a warning sign.

In [ ]:
pd.DataFrame([
    fit_eval(tr, va, te, feature_columns(X.columns, fs), fs) for fs in ("full", "pretxn", "no_balance")
]).round(4)

## Experiment 3: precision and recall at different false-alert budgets

The operating point is a business choice: a tighter false-alert budget means fewer investigator hours wasted but more missed fraud.

In [ ]:
cols = feature_columns(X.columns, "pretxn")
X_tr, y_tr = downsample_negatives(X.loc[tr, cols], y[tr], NEG_FRAC)
model = make_xgb(**PARAMS).fit(X_tr, y_tr)
p_val = prior_correct(model.predict_proba(X.loc[va, cols])[:, 1], NEG_FRAC)
p_te = prior_correct(model.predict_proba(X.loc[te, cols])[:, 1], NEG_FRAC)

rows = []
for budget in [0.0001, 0.0005, 0.001, 0.005]:
    thr = threshold_for_fpr(y[va], p_val, budget)
    op = operating_point(y[te], p_te >= thr, amount[te])
    rows.append({"FPR budget": f"{budget:.2%}", "recall": op["recall"], "precision": op["precision"],
                 "alerts": op["alerts"], "fraud amount caught %": op["fraud_amount_caught_pct"] * 100})
pd.DataFrame(rows).round(3)

## Experiment 4: model vs simple AML rules

In [ ]:
t = df.loc[X.index, "type"].astype(str).values[te]
old_org = df.loc[X.index, "oldbalanceOrg"].values[te]
rules = {
    "built-in isFlaggedFraud": df.loc[X.index, "isFlaggedFraud"].values[te] == 1,
    "account drain (amount >= old balance)": np.isin(t, ["TRANSFER", "CASH_OUT"]) & (old_org > 0) & (amount[te] >= old_org),
}
rows = []
for name, flag in rules.items():
    op = operating_point(y[te], flag, amount[te])
    rows.append({"detector": name, "recall": op["recall"], "precision": op["precision"], "alerts": op["alerts"]})
thr = threshold_for_fpr(y[va], p_val, 0.0005)
op = operating_point(y[te], p_te >= thr, amount[te])
rows.append({"detector": "model at 0.05% FPR budget", "recall": op["recall"], "precision": op["precision"], "alerts": op["alerts"]})
pd.DataFrame(rows).round(4)

A simple rule can have high recall but floods analysts with false alerts. The model's value is the precision it keeps at a similar recall.

## Final tuned model

The reported numbers come from `python -m src.train`. If you have run it, they are shown here.

In [ ]:
if METRICS_PATH.exists():
    m = json.loads(METRICS_PATH.read_text())
    print(pd.DataFrame(m["comparison"]).T[["roc_auc", "pr_auc"] + [f"recall_at_fpr_{b}" for b in FPR_BUDGETS]].round(4))
    op = m["main_operating_point"]
    print(f"\nMain operating point: recall {op['recall']:.1%}, precision {op['precision']:.1%}, alerts {op['alerts']:,}")
else:
    print("Run `python -m src.train` from the repository root to create models/metrics.json")